# Worked Project: California Housing, End to End

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/03_project.ipynb)

Runnable examples from **Data/ML Handbook** — section `03_project`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/03_project/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/proj01_make_csv.py`

In [ ]:
# Build a realistic "raw" CSV from the California Housing census data (20,640 block groups).
# We add the kinds of defects real exports have, so the cleaning steps are meaningful.
import numpy as np, pandas as pd
from sklearn.datasets import fetch_california_housing
rng = np.random.default_rng(42)
df = fetch_california_housing(as_frame=True).frame            # 8 features + MedHouseVal (in $100k)
df["region"] = np.select([df.Latitude > 37.2, df.Latitude > 35.5, df.Longitude > -118.3],
                         ["NorCal", "Central", "SoCal-Inland"], default="SoCal-Coast")
df.loc[rng.random(len(df)) < 0.03, "MedInc"] = np.nan          # 3% missing income
df.loc[rng.random(len(df)) < 0.01, "HouseAge"] = np.nan        # 1% missing age
df["region"] = df["region"].where(rng.random(len(df)) > 0.02, df["region"].str.lower() + " ")  # messy labels
df = pd.concat([df, df.sample(150, random_state=1)])           # 150 duplicate rows
df.to_csv("data/housing_raw.csv", index=False)
print(df.shape); print(df.head(3).round(2).to_string())

### `code/proj02_eda.py`

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/housing_raw.csv")
print("shape:", df.shape, "| duplicates:", df.duplicated().sum())
print("missing %:\n", (df.isna().mean() * 100).round(2)[lambda s: s > 0].to_string())
print("region labels:", sorted(df.region.unique()))
print(df.describe().T[["mean", "std", "min", "50%", "max"]].round(2).to_string())
print("target capped at 5.0 (= $500k):", (df.MedHouseVal >= 5.0).mean().round(3))
print("corr with target:\n", df.corr(numeric_only=True)["MedHouseVal"].drop("MedHouseVal").sort_values().round(3).to_string())

### `code/proj03_model.py`

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, make_column_selector as sel
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import lightgbm as lgb

df = pd.read_csv("data/housing_raw.csv").drop_duplicates()                 # 1. dedupe
df["region"] = df["region"].str.strip().str.replace("socal", "SoCal").str.replace("norcal", "NorCal").str.replace("central", "Central")
df["region"] = df["region"].str.replace("-inland", "-Inland").str.replace("-coast", "-Coast")
from housing_feats import add_feats   # lives in a module: pickled pipelines must import it at load time
y = df.pop("MedHouseVal"); X = df
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)   # 3. split BEFORE fitting anything

num = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())])
pre = ColumnTransformer([("num", num, sel(dtype_include="number")),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), ["region"])])
make = lambda model: Pipeline([("feats", FunctionTransformer(add_feats)), ("pre", pre), ("model", model)])
models = {"Baseline (mean)": make(DummyRegressor()),
          "Ridge": make(RidgeCV(alphas=np.logspace(-3, 3, 13))),
          "Random forest": make(RandomForestRegressor(300, min_samples_leaf=2, n_jobs=-1, random_state=0)),
          "HistGradientBoosting": make(HistGradientBoostingRegressor(max_iter=500, learning_rate=0.08, random_state=0)),
          "LightGBM": make(lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=63, subsample=0.8,
                                             subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=0))}
cv = KFold(5, shuffle=True, random_state=0)
rows = []
for name, m in models.items():
    r = cross_validate(m, X_tr, y_tr, cv=cv, n_jobs=1, scoring=("neg_root_mean_squared_error", "r2"))
    rows.append((name, -r["test_neg_root_mean_squared_error"].mean(), r["test_neg_root_mean_squared_error"].std(), r["test_r2"].mean()))
print(pd.DataFrame(rows, columns=["model", "CV RMSE", "± std", "CV R²"]).round(3).to_string(index=False))

### `code/proj04_tune_eval.py`

In [ ]:
import optuna, joblib
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
optuna.logging.set_verbosity(optuna.logging.WARNING)
def objective(trial):
    m = make(lgb.LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 300, 1500, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100, log=True),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        subsample=0.8, subsample_freq=1, verbose=-1, random_state=0))
    return -cross_val_score(m, X_tr, y_tr, cv=3, scoring="neg_root_mean_squared_error").mean()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=25)
print("best CV RMSE:", round(study.best_value, 4)); print("best params:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in study.best_params.items()})
final = make(lgb.LGBMRegressor(**study.best_params, subsample=0.8, subsample_freq=1, verbose=-1, random_state=0)).fit(X_tr, y_tr)
pred = final.predict(X_te)                                                 # test set touched ONCE
print(f"TEST  RMSE={root_mean_squared_error(y_te, pred):.4f}  MAE={mean_absolute_error(y_te, pred):.4f}  R²={r2_score(y_te, pred):.4f}")
print(f"=> typical error ≈ ${mean_absolute_error(y_te, pred)*100_000:,.0f} per block group")
pi = permutation_importance(final, X_te, y_te, n_repeats=5, random_state=0, scoring="neg_root_mean_squared_error")
print(pd.Series(pi.importances_mean, X_te.columns).sort_values(ascending=False).round(3).head(6).to_string())
joblib.dump(final, "data/house_model.joblib")

### `code/proj05_serve.py`

In [ ]:
import joblib, pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

model = joblib.load("data/house_model.joblib")
app = FastAPI(title="house-price")
class Block(BaseModel):
    MedInc: float = Field(gt=0); HouseAge: float; AveRooms: float; AveBedrms: float
    Population: float; AveOccup: float; Latitude: float; Longitude: float; region: str
@app.post("/predict")
def predict(b: Block):
    yhat = float(model.predict(pd.DataFrame([b.model_dump()]))[0])
    return {"median_value_usd": round(yhat * 100_000), "model_version": "lgbm-v1"}

client = TestClient(app)                                  # in-process test, no server needed
r = client.post("/predict", json=dict(MedInc=8.3, HouseAge=41, AveRooms=6.98, AveBedrms=1.02, Population=322,
                                     AveOccup=2.55, Latitude=37.88, Longitude=-122.23, region="NorCal"))
print(r.status_code, r.json())
bad = client.post("/predict", json=dict(MedInc=-1, HouseAge=41, AveRooms=6.98, AveBedrms=1.02, Population=322,
                                       AveOccup=2.55, Latitude=37.88, Longitude=-122.23, region="NorCal"))
print(bad.status_code, "validation error ->", bad.json()["detail"][0]["msg"])